In [ ]:
from google.colab import drive
import os
import zipfile

drive.mount('/content/drive')

# --- UPDATED PATH ---
ZIP_PATH = "/content/drive/MyDrive/data/banana_dataset_complete_FINAL (1).zip"

LOCAL_DATA_DIR = "/content/banana_dataset"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(LOCAL_DATA_DIR, 'train')):
    print(f"Found zip at: {ZIP_PATH}")
    print("Extracting dataset to local SSD (takes ~1-2 min)...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_DIR)
    print("Extraction complete!")
else:
    print("Dataset already extracted locally.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found zip at: /content/drive/MyDrive/data/banana_dataset_complete_FINAL (1).zip
Extracting dataset to local SSD (takes ~1-2 min)...
Extraction complete!


In [ ]:
import numpy as np
import pandas as pd
import cv2
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

CACHE_DIR = "/content/matrix_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

matrix_tasks = []

for split in ['train', 'val', 'test']:
    for stage in os.listdir(os.path.join(LOCAL_DATA_DIR, split)):
        stage_path = os.path.join(LOCAL_DATA_DIR, split, stage)
        mat_dir = os.path.join(stage_path, 'Thermal_Matrix')

        if not os.path.exists(mat_dir): continue

        for f in os.listdir(mat_dir):
            if f.endswith('.xlsx'):
                src = os.path.join(mat_dir, f)
                dst = os.path.join(CACHE_DIR, f.replace('.xlsx', '.npy'))
                if not os.path.exists(dst):
                    matrix_tasks.append((src, dst))

print(f"Found {len(matrix_tasks)} matrices to convert. Starting...")

def convert_one(task):
    src, dst = task
    try:
        df = pd.read_excel(src, header=None)
        arr = df.applymap(lambda x: float(str(x).replace('°C', '')) if pd.notna(x) else np.nan).values
        arr = np.nan_to_num(arr, nan=0.0)
        arr = cv2.resize(arr, (192, 256), interpolation=cv2.INTER_LINEAR)
        np.save(dst, arr.astype(np.float32))
        return True
    except Exception as e:
        return False

success = 0
with ThreadPoolExecutor(max_workers=8) as ex:
    for res in tqdm(ex.map(convert_one, matrix_tasks), total=len(matrix_tasks)):
        if res: success += 1

print(f"Successfully converted {success} matrices to .npy")

Found 2961 matrices to convert. Starting...


  0%|          | 0/2961 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/tmp/ipykernel_576/2602247422.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  arr = df.applymap(lambda x: float(str(x).replace('°C', '')) if pd.notna(x) else np.nan).values
/tmp/ipykernel_576/2602247422.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  arr = df.applymap(lambda x: float(str(x).replace('°C', '')) if pd.notna(x) else np.nan).values
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/tmp/ipykernel_576/2602247422.py:32: FutureWarning: DataFrame.applymap has be

Successfully converted 2961 matrices to .npy


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ['Stage_1_Mature_Green', 'Stage_2_Green_Yellow_Transition',
               'Stage_3_Early_Ripening', 'Stage_4_Fully_Yellow', 'Stage_5_Yellow_Spotted']
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
IMG_SIZE = 224

class BananaDataset(Dataset):
    def __init__(self, root_dir, split, train=False):
        self.samples = []
        self.train = train

        split_path = os.path.join(root_dir, split)

        for stage in CLASS_NAMES:
            stage_path = os.path.join(split_path, stage)
            if not os.path.exists(stage_path): continue

            rgb_dir = os.path.join(stage_path, 'RGB')
            th_dir = os.path.join(stage_path, 'Thermal')
            mat_dir = os.path.join(stage_path, 'Thermal_Matrix')

            # Sort to align the sequential filenames
            rgb_files = sorted(os.listdir(rgb_dir))
            th_files = sorted(os.listdir(th_dir))
            mat_files = sorted(os.listdir(mat_dir))

            # Align by shortest length to prevent index errors
            num = min(len(rgb_files), len(th_files), len(mat_files))

            for i in range(num):
                self.samples.append({
                    'rgb': os.path.join(rgb_dir, rgb_files[i]),
                    'th': os.path.join(th_dir, th_files[i]),
                    'mat': os.path.join(CACHE_DIR, mat_files[i].replace('.xlsx', '.npy')),
                    'label': CLASS_TO_IDX[stage]
                })

        self.train_tf = T.Compose([
            T.Resize((256, 256)),
            T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.1, 0.1, 0.1),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
        self.val_tf = T.Compose([
            T.Resize((256, 256)),
            T.CenterCrop(IMG_SIZE),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        rgb = Image.open(s['rgb']).convert('RGB')
        th = Image.open(s['th']).convert('L').resize((IMG_SIZE, IMG_SIZE)).convert('RGB')

        rgb = self.train_tf(rgb) if self.train else self.val_tf(rgb)
        th = self.train_tf(th) if self.train else self.val_tf(th)

        mat = np.load(s['mat'])
        mat = torch.tensor(mat, dtype=torch.float32).unsqueeze(0)
        mat = (mat - mat.mean()) / (mat.std() + 1e-6)
        mat = torch.nn.functional.interpolate(mat.unsqueeze(0), size=(64, 48), mode='bilinear').squeeze(0)

        return rgb, th, mat, torch.tensor(s['label'], dtype=torch.long)

train_ds = BananaDataset(LOCAL_DATA_DIR, 'train', train=True)
val_ds = BananaDataset(LOCAL_DATA_DIR, 'val', train=False)
test_ds = BananaDataset(LOCAL_DATA_DIR, 'test', train=False)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

labels = [s['label'] for s in train_ds.samples]
cnt = Counter(labels)
weights = [1.0 / cnt[l] for l in labels]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

dl_train = DataLoader(train_ds, batch_size=16, sampler=sampler, num_workers=2, pin_memory=True)
dl_val = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
dl_test = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# Smoke test
rgb, th, mat, y = next(iter(dl_train))
print(f"Shapes - RGB: {rgb.shape}, TH: {th.shape}, MAT: {mat.shape}, Y: {y.shape}")

Train: 2038 | Val: 452 | Test: 469
Shapes - RGB: torch.Size([16, 3, 224, 224]), TH: torch.Size([16, 3, 224, 224]), MAT: torch.Size([16, 1, 64, 48]), Y: torch.Size([16])


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

class EfficientNetBranch(nn.Module):
    def __init__(self, in_ch=3, out_dim=512):
        super().__init__()
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        if in_ch != 3:
            old = m.features[0][0]
            m.features[0][0] = nn.Conv2d(in_ch, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
        for p in m.features.parameters(): p.requires_grad = False
        self.backbone = m.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Linear(m.classifier[1].in_features, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(), nn.Dropout(0.3))

    def forward(self, x):
        with torch.no_grad(): f = self.pool(self.backbone(x)).flatten(1)
        return self.proj(f)

class MatrixBranch(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, out_dim), nn.ReLU()
        )
    def forward(self, x): return self.net(x)

class BananaNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb = EfficientNetBranch(3, 512)
        self.th = EfficientNetBranch(3, 512)
        self.mat = MatrixBranch(256)
        self.classifier = nn.Sequential(nn.Linear(512+512+256, 512), nn.ReLU(), nn.Dropout(0.4), nn.Linear(512, 5))
    def forward(self, rgb, th, mat):
        f = torch.cat([self.rgb(rgb), self.th(th), self.mat(mat)], dim=1)
        return self.classifier(f)

model = BananaNet().to(DEVICE)
opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
crit = nn.CrossEntropyLoss()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 102MB/s]


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import time

EPOCHS = 20
best_f1 = 0
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', patience=3, factor=0.5)

for epoch in range(1, EPOCHS+1):
    model.train()
    tr_preds, tr_labels = [], []
    start = time.time()

    for rgb, th, mat, y in dl_train:
        rgb, th, mat, y = rgb.to(DEVICE), th.to(DEVICE), mat.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits = model(rgb, th, mat)
        loss = crit(logits, y)
        loss.backward()
        opt.step()

        tr_preds.extend(logits.argmax(1).detach().cpu().numpy())
        tr_labels.extend(y.cpu().numpy())

    tr_acc = accuracy_score(tr_labels, tr_preds)

    model.eval()
    v_preds, v_labels = [], []
    with torch.no_grad():
        for rgb, th, mat, y in dl_val:
            rgb, th, mat = rgb.to(DEVICE), th.to(DEVICE), mat.to(DEVICE)
            out = model(rgb, th, mat)
            v_preds.extend(out.argmax(1).cpu().numpy())
            v_labels.extend(y.numpy())

    va_acc = accuracy_score(v_labels, v_preds)
    va_f1 = f1_score(v_labels, v_preds, average='macro', zero_division=0)
    scheduler.step(va_f1)

    print(f"Epoch {epoch:02d} | Train Acc: {tr_acc:.3f} | Val Acc: {va_acc:.3f} | Val F1: {va_f1:.4f} | Time: {time.time()-start:.1f}s")

    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save(model.state_dict(), "/content/best_ripeness_model.pth")
        print("  ↳ Saved best model")

print(f"Training complete. Best Val F1: {best_f1:.4f}")

Epoch 01 | Train Acc: 0.825 | Val Acc: 0.861 | Val F1: 0.8458 | Time: 357.0s
  ↳ Saved best model
Epoch 02 | Train Acc: 0.880 | Val Acc: 0.845 | Val F1: 0.8446 | Time: 350.4s
Epoch 03 | Train Acc: 0.910 | Val Acc: 0.856 | Val F1: 0.8618 | Time: 340.5s
  ↳ Saved best model
Epoch 04 | Train Acc: 0.921 | Val Acc: 0.872 | Val F1: 0.8822 | Time: 352.5s
  ↳ Saved best model
Epoch 05 | Train Acc: 0.919 | Val Acc: 0.861 | Val F1: 0.8680 | Time: 344.0s
Epoch 06 | Train Acc: 0.921 | Val Acc: 0.854 | Val F1: 0.8660 | Time: 345.6s
Epoch 07 | Train Acc: 0.925 | Val Acc: 0.830 | Val F1: 0.8379 | Time: 347.0s
Epoch 08 | Train Acc: 0.921 | Val Acc: 0.865 | Val F1: 0.8718 | Time: 350.6s
Epoch 09 | Train Acc: 0.934 | Val Acc: 0.876 | Val F1: 0.8782 | Time: 345.8s
Epoch 10 | Train Acc: 0.938 | Val Acc: 0.854 | Val F1: 0.8629 | Time: 339.6s
Epoch 11 | Train Acc: 0.943 | Val Acc: 0.863 | Val F1: 0.8773 | Time: 345.5s
Epoch 12 | Train Acc: 0.946 | Val Acc: 0.905 | Val F1: 0.9123 | Time: 352.2s
  ↳ Saved bes

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

model.load_state_dict(torch.load("/content/best_ripeness_model.pth"))
model.eval()

t_preds, t_labels = [], []
with torch.no_grad():
    for rgb, th, mat, y in dl_test:
        rgb, th, mat = rgb.to(DEVICE), th.to(DEVICE), mat.to(DEVICE)
        out = model(rgb, th, mat)
        t_preds.extend(out.argmax(1).cpu().numpy())
        t_labels.extend(y.numpy())

print("=== TEST RESULTS ===")
print(classification_report(t_labels, t_preds, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(t_labels, t_preds)
print("\nConfusion Matrix:")
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

=== TEST RESULTS ===
                                 precision    recall  f1-score   support

           Stage_1_Mature_Green     0.4348    0.3333    0.3774        30
Stage_2_Green_Yellow_Transition     1.0000    0.6591    0.7945        88
         Stage_3_Early_Ripening     0.7727    0.9745    0.8620       157
           Stage_4_Fully_Yellow     0.9394    0.8986    0.9185       138
         Stage_5_Yellow_Spotted     0.9483    0.9821    0.9649        56

                       accuracy                         0.8529       469
                      macro avg     0.8190    0.7695    0.7835       469
                   weighted avg     0.8638    0.8529    0.8472       469


Confusion Matrix:
                                 Stage_1_Mature_Green  \
Stage_1_Mature_Green                               10   
Stage_2_Green_Yellow_Transition                    11   
Stage_3_Early_Ripening                              1   
Stage_4_Fully_Yellow                                1   
Stage_5_Yellow_

In [ ]:
import shutil
shutil.copy("/content/best_ripeness_model.pth",
            "/content/drive/MyDrive/best_ripeness_model.pth")
print("Saved to Drive.")

Saved to Drive.


In [ ]:
import os
for p in ["/content/drive/MyDrive/best_ripeness_model.pth",
          "/content/drive/MyDrive/best_banana_multimodal.pth"]:
    print(p, "→", os.path.exists(p),
          f"{os.path.getsize(p)/1e6:.1f} MB" if os.path.exists(p) else "")

/content/drive/MyDrive/best_ripeness_model.pth → True 40.7 MB
/content/drive/MyDrive/best_banana_multimodal.pth → True 39.4 MB
